# Movie Recommendation System

A content-based movie recommender built with the TMDB 5000 Movies and Credits datasets.

**What this notebook does**
1. Loads and validates the datasets.
2. Extracts genres, keywords, the top three cast members, and directors.
3. Combines movie metadata into a single text field (`tags`).
4. Converts tags into numerical features with `CountVectorizer`.
5. Recommends movies using cosine similarity.

## 1. Imports and configuration

In [1]:
from pathlib import Path
import ast
import pickle

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA_DIR = Path(".")
MOVIES_PATH = DATA_DIR / "tmdb_5000_movies.csv"
CREDITS_PATH = DATA_DIR / "tmdb_5000_credits.csv"

RANDOM_STATE = 42

## 2. Load and inspect the data

In [2]:
for path in (MOVIES_PATH, CREDITS_PATH):
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path.resolve()}. Put the CSV file there or update DATA_DIR."
        )

movies = pd.read_csv(MOVIES_PATH)
credits = pd.read_csv(CREDITS_PATH)

print(f"Movies dataset:  {movies.shape[0]:,} rows × {movies.shape[1]} columns")
print(f"Credits dataset: {credits.shape[0]:,} rows × {credits.shape[1]} columns")

display(movies.head(3))
display(credits.head(3))

Movies dataset:  4,803 rows × 20 columns
Credits dataset: 4,803 rows × 4 columns


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466


,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."


## 3. Merge and select relevant columns

The merge uses `title`, as in the original notebook. We then keep only fields used to build movie profiles.

In [3]:
required_movie_cols = {"id", "title", "overview", "genres", "keywords"}
required_credit_cols = {"title", "cast", "crew"}

missing_movies = required_movie_cols - set(movies.columns)
missing_credits = required_credit_cols - set(credits.columns)
if missing_movies or missing_credits:
    raise ValueError(
        f"Unexpected dataset columns. Missing from movies: {missing_movies}; "
        f"missing from credits: {missing_credits}"
    )

# Rename the movie dataset's id to movie_id for clarity.
movies = movies.rename(columns={"id": "movie_id"})
credits = credits[["title", "cast", "crew"]]

movies = movies.merge(credits, on="title", how="inner", validate="many_to_many")
movies = movies[["movie_id", "title", "overview", "genres", "keywords", "cast", "crew"]].copy()

# Rows without enough text/metadata cannot produce a useful content profile.
movies = movies.dropna(subset=["title", "overview", "genres", "keywords", "cast", "crew"]).copy()
movies = movies.drop_duplicates(subset=["movie_id"]).reset_index(drop=True)

print(f"Rows retained after merge and cleaning: {len(movies):,}")
display(movies.head(3))

Rows retained after merge and cleaning: 4,800


,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."


## 4. Parse the metadata

The `genres`, `keywords`, `cast`, and `crew` columns contain Python-literal-style lists encoded as strings. The helper below safely parses them and returns an empty list for malformed or missing values.

In [4]:
def parse_metadata(value):
    """Parse a stringified list of dictionaries; return [] if parsing fails."""
    if not isinstance(value, str) or not value.strip():
        return []
    try:
        parsed = ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []
    return parsed if isinstance(parsed, list) else []


def get_names(value, limit=None):
    """Extract non-empty 'name' values from a metadata field."""
    names = []
    for item in parse_metadata(value):
        if isinstance(item, dict):
            name = item.get("name")
            if isinstance(name, str) and name.strip():
                names.append(name.strip())
        if limit is not None and len(names) >= limit:
            break
    return names


def get_directors(value):
    """Extract director names from the crew metadata."""
    directors = []
    for item in parse_metadata(value):
        if isinstance(item, dict) and item.get("job") == "Director":
            name = item.get("name")
            if isinstance(name, str) and name.strip():
                directors.append(name.strip())
    return directors


for column in ["genres", "keywords"]:
    movies[column] = movies[column].apply(get_names)

movies["cast"] = movies["cast"].apply(lambda value: get_names(value, limit=3))
movies["crew"] = movies["crew"].apply(get_directors)

display(movies[["title", "genres", "keywords", "cast", "crew"]].head(5))

,title,genres,keywords,cast,crew
0,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]
2,Spectre,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes]
3,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman]",[Christopher Nolan]
4,John Carter,"[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton]",[Andrew Stanton]


## 5. Build a combined tag field

Whitespace is removed inside names so multi-word names such as `Sam Worthington` become `SamWorthington`. This helps the vectorizer treat each name as one feature instead of separate words. The original display title is left unchanged.

In [5]:
def normalize_names(items):
    return [item.replace(" ", "") for item in items if isinstance(item, str) and item.strip()]


for column in ["genres", "keywords", "cast", "crew"]:
    movies[column] = movies[column].apply(normalize_names)

movies["overview"] = movies["overview"].fillna("").astype(str).str.split()

tag_columns = ["overview", "genres", "keywords", "cast", "crew"]
movies["tags"] = movies[tag_columns].apply(
    lambda row: " ".join(token for values in row for token in values),
    axis=1,
)

# Keep a compact table for recommendation and display.
movie_list = movies[["movie_id", "title", "tags"]].copy()
movie_list["tags"] = movie_list["tags"].str.strip()
movie_list = movie_list.reset_index(drop=True)

display(movie_list.head())

,movie_id,title,tags
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...
4,49529,John Carter,"John Carter is a war-weary, former military ca..."


## 6. Convert tags into features

`CountVectorizer` creates a bag-of-words representation. Common English stop words are removed, and the vocabulary is capped at 5,000 features.

The feature matrix remains sparse to avoid unnecessarily allocating a large dense array.

In [6]:
vectorizer = CountVectorizer(max_features=5000, stop_words="english")
tag_vectors = vectorizer.fit_transform(movie_list["tags"])

print(f"Feature matrix shape: {tag_vectors.shape}")
print(f"Non-zero feature values: {tag_vectors.nnz:,}")

Feature matrix shape: (4800, 5000)
Non-zero feature values: 133,704


## 7. Recommend similar movies

Cosine similarity compares the tag vectors. Instead of calculating and storing every movie-to-movie similarity in advance, this function computes similarities only for the requested movie.

The title lookup is case-insensitive. If multiple movies share the same title, the function shows the available matches and uses the first one unless `match_index` is supplied.

In [7]:
def recommend(movie_title, n_recommendations=5, match_index=0):
    """Return the most similar movies as a DataFrame.

    Parameters
    ----------
    movie_title : str
        Movie title to look up (case-insensitive).
    n_recommendations : int
        Number of recommendations to return.
    match_index : int
        Which matching title to use when titles are duplicated.
    """
    if not isinstance(movie_title, str) or not movie_title.strip():
        raise ValueError("Enter a non-empty movie title.")
    if n_recommendations < 1:
        raise ValueError("n_recommendations must be at least 1.")

    matches = movie_list.index[
        movie_list["title"].str.casefold() == movie_title.strip().casefold()
    ].tolist()

    if not matches:
        close_matches = movie_list.loc[
            movie_list["title"].str.contains(movie_title.strip(), case=False, na=False),
            "title",
        ].drop_duplicates().head(10).tolist()
        suggestion_text = f" Similar title matches: {close_matches}" if close_matches else ""
        raise ValueError(f"Movie not found: {movie_title!r}.{suggestion_text}")

    if not 0 <= match_index < len(matches):
        raise IndexError(f"match_index must be between 0 and {len(matches) - 1}.")

    movie_index = matches[match_index]
    scores = cosine_similarity(tag_vectors[movie_index], tag_vectors).ravel()
    ranked_indices = scores.argsort()[::-1]

    # Exclude the selected movie itself.
    ranked_indices = [idx for idx in ranked_indices if idx != movie_index]
    ranked_indices = ranked_indices[:n_recommendations]

    result = movie_list.iloc[ranked_indices][["movie_id", "title"]].copy()
    result["similarity_score"] = scores[ranked_indices]
    return result.reset_index(drop=True)

### Try it

In [8]:
recommend("The Lego Movie", n_recommendations=5)

,movie_id,title,similarity_score
0,17711,The Adventures of Rocky & Bullwinkle,0.295918
1,9975,Curious George,0.287368
2,76285,Percy Jackson: Sea of Monsters,0.280000
3,170687,The Boxtrolls,0.280000
4,49519,The Croods,0.269680


## 8. Save the model artifacts

Save the movie table, fitted vectorizer, and sparse feature matrix. This avoids storing a full dense similarity matrix and allows recommendations to be generated later without retraining.

**Security note:** only load pickle files that you created or trust. Pickle files can execute code when loaded.

In [9]:
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

with (ARTIFACT_DIR / "movie_list.pkl").open("wb") as file:
    pickle.dump(movie_list, file)

with (ARTIFACT_DIR / "vectorizer.pkl").open("wb") as file:
    pickle.dump(vectorizer, file)

with (ARTIFACT_DIR / "tag_vectors.pkl").open("wb") as file:
    pickle.dump(tag_vectors, file)

print(f"Saved model artifacts to: {ARTIFACT_DIR.resolve()}")

Saved model artifacts to: C:\Files\Learning\Projects\Movies-Recommender-System\artifacts


## Possible next improvements

- Add fuzzy title matching for typos (for example, with `rapidfuzz`).
- Tune how much the overview, genres, cast, and keywords influence similarity.
- Evaluate recommendations with a small manually reviewed test set.
- If you deploy this as an app, load the saved artifacts once at startup rather than rebuilding them for every request.